## Imports

In [5]:
# Python standards
import numpy as np
import pandas as pd 
import seaborn as sns
import os
import csv
import mygene

# Project Specific
import mygene
from Bio import Entrez

## Import Data

In [6]:
# Import data files - be able to differentiate between .csv and txt

# Makes index column (row labels) lowercase and string format
def lowercase_index(table):
    table.index = table.index.map(str)
    table.index = table.index.str.lower()
    table = table.astype(int)
    return table

# Turn input data into pandas tables
def make_table(filepath):
    filename, extension = os.path.splitext(filepath)

    # Convert .csv file to 
    if extension == ".csv":
        table = pd.read_csv(filepath, header=0, index_col=0) 

    # For non .csv files (.txt, .tsv), use Sniffer to automatically detect delimiter type
    elif extension in [".txt", ".tsv"]:
        with open(filepath, 'r') as f1:
            dialect = csv.Sniffer().sniff(f1.readline())
            delimiter = dialect.delimiter
        table = pd.read_csv(filepath, sep=delimiter, header=0, index_col=0)

    # Raise error for non-supported file types
    else:
        raise TypeError("Unsupported format - file must be .csv, .txt, or .tsv")

    # Convert index (row names) to strings and all lowercase - this makes future processing/matching much easier  
    table = lowercase_index(table)
    
    return table

In [7]:
gse167216 = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE167216_Raw_gene_counts_matrix.txt")
gse167216

,I00001,I00002,I00003,I00004,I00005,I00006,I00007,I00008,I00009,I00010,...,I00027,I00028,I00029,I00030,I00031,I00032,I00033,I00034,I00035,I00036
0610007p14rik,858,901,1053,1413,882,704,947,747,1253,928,...,1108,1293,1510,1080,474,1098,857,1187,1572,1111
0610009b22rik,430,515,450,423,395,341,392,449,556,382,...,395,428,515,522,625,487,332,406,440,459
0610009l18rik,14,10,12,9,8,8,12,11,8,5,...,13,7,10,8,10,8,6,7,20,13
0610009o20rik,600,462,559,541,441,268,459,523,594,499,...,475,441,644,479,619,457,437,508,690,378
0610010f05rik,615,439,503,635,448,399,587,592,663,598,...,665,621,733,546,791,572,587,548,701,539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd3,7111,12690,9805,8066,6143,6600,4815,9508,8672,9020,...,6218,9097,6992,5356,10781,8872,8342,8975,6406,3930
mt-nd4,21825,29177,26462,33494,32804,16978,15866,33967,30352,29477,...,33113,29928,24068,16664,35913,29747,25289,41894,30768,16901
mt-nd4l,2540,3058,2926,3801,2663,1873,1980,3266,3416,3179,...,3703,3160,2709,1830,3963,3257,2984,3175,3554,2027
mt-nd5,12329,19694,17297,17376,22612,10367,9914,23960,17992,18814,...,21511,15814,15392,10895,23150,15632,16334,31330,17930,9719


In [8]:
gse130970 = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE130970_all_sample_salmon_tximport_counts_entrez_gene_ID.csv")
gse130970

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
entrez_id,,,,,,,,,,,,,,,,,,,,,
1,15968,15623,12255,13328,6906,10556,9997,10990,14610,8831,...,12238,12415,8589,5733,11558,10913,7998,11437,12920,12134
10,1898,1635,1476,1359,847,2195,1546,2677,1094,1251,...,1058,1173,1389,806,1151,1614,1001,1216,1806,1381
100,100,72,67,70,112,47,76,80,57,37,...,34,49,30,22,41,35,38,59,38,28
1000,2969,2997,2547,2625,3644,2485,1216,2240,2569,2756,...,3035,2525,2624,2017,2851,2290,1854,2906,2652,2712
10000,500,398,526,587,907,327,442,408,500,394,...,521,501,296,326,374,405,439,644,704,436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,747,655,491,579,668,477,212,519,496,663,...,648,601,734,549,691,554,433,637,713,567
9991,2211,1761,2105,1730,3392,1700,1052,1839,1679,1464,...,1203,1795,1140,873,1204,1427,1740,1900,1616,1388
9992,107,10,11,27,5,8,31,11,6,10,...,11,16,15,9,24,82,16,13,19,18


In addition, each dataframe must have an accompanying metadata dataframe, containing information about the `group` (species) and `condition` (disease status) of each sample. Therefore, `.csv` files will be imported that contain this information.

In [9]:
gse130970_md = pd.read_csv("GSE130970_Metadata.csv", header=0, index_col=0)
gse167216_md = pd.read_csv("GSE167216_Metadata.csv", header=0, index_col=0)
gse130970_md

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
condition,5,4,5,5,6,6,5,6,4,4,...,1,1,0,0,0,6,6,5,3,0
group,human,human,human,human,human,human,human,human,human,human,...,human,human,human,human,human,human,human,human,human,human


## Data formatting

For the dataset GSE130970, the gene IDs are instead recorded as `entrez_id`. We want to map these to gene names, so we use the `mygene` package to query the Entrez ID system for their matching gene symbols, creating a new pandas dataframe with these symbols.

In [10]:
def convert_entrez(table, genome):
    
    mg = mygene.MyGeneInfo()
    entrez_ID_list = table.index.tolist()
    
    print("Starting MyGene.info query...")
    
    results = mg.querymany(entrez_ID_list,
                           scopes='entrezgene',
                           fields='symbol',
                           species=genome,
                           as_dataframe=True)
    
    # Drop entries without corresponding gene symbols 
    results = results.dropna(subset=['symbol'])
    mapping = results[['symbol']]
    mapping = mapping[~mapping.index.duplicated(keep='first')]
    
    # Find where mapping index intersects with original Entrez ID table and overwrite - replaces labels
    table = table.loc[table.index.intersection(mapping.index)]
    table.index = mapping.loc[table.index, 'symbol']
    
    # Duplicate gene symbols have their expression patterns averaged
    table = table.groupby(table.index).mean()
    
    print(f"Mapped {len(table)} genes to symbols")

    # Convert index (row names) to strings and all lowercase  
    table = lowercase_index(table)
    
    return table

In [11]:
gse130970 = convert_entrez(gse130970, "human")
gse130970

Starting MyGene.info query...


159 input query terms found no hit:	['100126582', '100127889', '100128374', '100130285', '100132705', '100133144', '100133301', '1001340


Mapped 19426 genes to symbols


,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
symbol,,,,,,,,,,,,,,,,,,,,,
a1bg,15968,15623,12255,13328,6906,10556,9997,10990,14610,8831,...,12238,12415,8589,5733,11558,10913,7998,11437,12920,12134
a1cf,21465,23403,18116,17159,9293,19942,10803,15732,18271,19720,...,17101,17088,18674,18042,19301,16481,17023,21535,25753,18947
a2m,85499,70829,97974,46332,79707,38391,59010,20908,105181,25295,...,65044,103781,43988,35546,89737,39190,36627,53106,52269,67573
a2ml1,93,94,88,107,128,56,31,36,92,103,...,88,77,66,126,100,82,73,89,122,86
a3galt2,0,0,2,0,0,0,6,0,1,1,...,0,2,0,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zyg11a,222,180,317,231,146,177,106,110,140,190,...,174,179,344,164,214,188,223,190,239,308
zyg11b,4064,2989,2904,3374,1993,2806,1431,3217,3090,3477,...,2217,2713,2586,2311,2320,3264,2567,3913,3630,2571
zyx,812,481,537,527,1194,619,369,501,292,275,...,250,300,141,153,193,203,397,338,265,155


In [12]:
# Save the completed tables to .csv files, in case they are needed later

gse130970.to_csv("GSE130970_all_sample_counts_gene_symbol.csv")
gse167216.to_csv("GSE167216_all_sample_counts_gene_symbol.csv")

In [13]:
"""
To use the package for differential gene expression analysis, 
we combine the dataframes for GSE130970 and GSE167216, and their 
accompanying metadata. However, the genes between each species do
not entirely overlap. We must therefore write code to sort and 
save the similar genes between the species. 
"""

def union_datasets(df1, df2, md1, md2):
    common_genes = df1.index.intersection(df2.index)

    # Create new 
    df1_sub = df1.loc[df1.index.intersection(common_genes)]
    df2_sub = df2.loc[df2.index.intersection(common_genes)]
    
    # Align ordering
    df1_sub = df1_sub.sort_index()
    df2_sub = df2_sub.sort_index()
    
    # Combine along columns (samples)
    combined_df = pd.concat([df1_sub, df2_sub], axis=1)
    combined_md = pd.concat([md1, md2], axis=1)

    return combined_df, combined_md

## Differential gene expression analysis

Now, we perform **differential gene expression analysis**. In the original paper, this was done in R using the `limma` package; to perform this in a Python environment, we can instead use the `PyDESeq2` package and pipeline. This starts with importing the `pydeseq2` package and its associated functions.
```bash
pip install pydeseq2
```

In [14]:
# Import pydeseq2 important packages

import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data # not necessary, used for tests

However, the PyDESeq2 package can only perform pairwise comparisons, but there might be multiple different disease or treatment types or states. Therefore, we must write code for Python that will automatically perform pairwise DESeq2 analysis between all conditions against a user-specified control variable.

In [15]:
# Function for deseq2
def run_deseq2(counts_df, metadata, condition_name, control_label):

    # Don't waste my computer power
    if control_label not in metadata[condition_name].unique():
        raise ValueError(f"Designated control label cannot be found in condition types")

    inference = DefaultInference(n_cpus=8)

    # Create a dds object for the analysis
    dds = DeseqDataSet(
        counts=counts_df,
        metadata=metadata,
        design=condition_name,
        refit_cooks=True,
        inference=inference, # n_cpus=8, # n_cpus can be specified here or in the inference object
    )

    # Run DESeq2 pipeline
    dds.deseq2() # the magic!
    results = {} # return variable - dictionary of different DeseqStats
    
    # Compute statistics of all versus control
    for condition in metadata[condition_name].unique():
        if condition != control_label:
            contrast_name = f"{condition}_vs_{control_label}"
            print(f"Running contrast: {contrast_name}")
            
            stat_res = DeseqStats(dds,contrast=(condition_name, condition, control_label))
            stat_res.summary()

            res_df = stat_res.results_df.copy()
            results[contrast_name] = res_df
    
    print(f"Differential expression complete: {results.shape[0]} comparisons made")

    return results

In [4]:
# gse130970_results = run_deseq2(gse130970.T,gse130970_md.T, condition_name="condition", control_label="0")

NameError: name 'gse130970' is not defined